<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Apache%20Spark/4-Machine_Learning_with_MLlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color:#E25822; font-family:Arial, sans-serif; font-size:2.4em; border-bottom: 3px solid #E25822; padding-bottom:10px;">
  Machine Learning with Apache Spark MLlib
</h1>

<p style="font-size:1.1em; color:#444;">
  In this notebook we explore <strong>Apache Spark MLlib</strong> — Spark's scalable machine learning library.
  We cover <em>regression</em>, <em>classification</em>, <em>clustering</em>, <em>ML Pipelines</em>, and
  <em>hyperparameter tuning</em> using real-world inspired datasets generated entirely inline.
</p>

| Topic | Algorithms |
|---|---|
| Regression | Linear Regression, Decision Tree Regressor |
| Classification | Logistic Regression, Random Forest Classifier |
| Clustering | K-Means |
| Pipelines | StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler → RF |
| Tuning | ParamGridBuilder, CrossValidator |

<h2 style="color:#E25822; font-family:Arial, sans-serif;">1. MLlib Overview</h2>

### What is MLlib?

**Apache Spark MLlib** is Spark's built-in machine learning library. It provides:

- **Scalability**: runs across hundreds of nodes on datasets that do not fit in memory on a single machine.
- **Unified API**: the same code works locally (for development) and on a cluster (for production).
- **DataFrame-based API** (`spark.ml`): the modern, recommended API built on DataFrames and Pipelines.
- **RDD-based API** (`spark.mllib`): the legacy API, still available but not actively developed.

### Supported Algorithm Families

| Family | Examples |
|---|---|
| **Regression** | Linear Regression, Decision Tree, Random Forest, Gradient Boosted Trees, Isotonic Regression |
| **Classification** | Logistic Regression, Decision Tree, Random Forest, GBT, Naïve Bayes, Linear SVM, MLP |
| **Clustering** | K-Means, Bisecting K-Means, Gaussian Mixture, LDA |
| **Recommendation** | ALS (Alternating Least Squares) |
| **Dimensionality Reduction** | PCA, SVD |
| **Feature Engineering** | VectorAssembler, StandardScaler, MinMaxScaler, StringIndexer, OneHotEncoder, Tokenizer, TF-IDF |

### The Pipeline Concept

A **Pipeline** chains multiple `Transformers` and `Estimators` into a single reusable workflow:

```
Raw Data
   │
   ▼
┌─────────────────┐
│  StringIndexer  │  ← Transformer: converts string labels to numeric indices
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ OneHotEncoder   │  ← Transformer: converts indices to binary vectors
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ VectorAssembler │  ← Transformer: merges columns into a single feature vector
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ StandardScaler  │  ← Estimator/Transformer: learns mean & std, then scales
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   ML Algorithm  │  ← Estimator: learns model parameters from training data
└────────┬────────┘
         │
         ▼
   Predictions
```

Calling `pipeline.fit(train_data)` triggers the entire sequence. The resulting `PipelineModel` can then be applied to new data with a single `transform()` call.

<h2 style="color:#E25822; font-family:Arial, sans-serif;">2. Environment Setup</h2>

### How is MLlib Similar to scikit-learn?

If you've used **scikit-learn** before, you'll find MLlib very familiar. The API is almost identical:

| Concept | scikit-learn | Spark MLlib |
|---|---|---|
| Import | `from sklearn.linear_model import LinearRegression` | `from pyspark.ml.regression import LinearRegression` |
| Create model | `model = LinearRegression()` | `model = LinearRegression()` |
| Fit on data | `model.fit(X_train, y_train)` | `model.fit(train_df)` |
| Make predictions | `predictions = model.predict(X_test)` | `predictions = model.transform(test_df)` |
| Evaluate | `from sklearn.metrics import mean_squared_error` | `from pyspark.ml.evaluation import RegressionEvaluator` |

**Key differences:**
- **scikit-learn**: Works on single machine (in-memory), fast for datasets that fit in RAM
- **Spark MLlib**: Works distributed across clusters, built for BIG DATA that doesn't fit on one machine
- **Input format**: scikit-learn uses NumPy arrays; MLlib uses DataFrames
- **Method names**: scikit-learn uses `.predict()`, MLlib uses `.transform()` (Transformers pattern)
- **Preprocessing**: scikit-learn has `.fit_transform()`, MLlib separates `.fit()` and `.transform()` to prevent data leakage


In [6]:
# Install PySpark (only needed on Colab or environments without PySpark)
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pyspark --quiet
    print("PySpark installed successfully.")
else:
    print("Not running on Colab — assuming PySpark is already installed.")

Not running on Colab — assuming PySpark is already installed.


### 2a. Create the SparkSession

After ensuring PySpark is installed, we import the required Spark modules and create a `SparkSession`. Key imports:

- **`SparkSession`** — the single entry point to Spark SQL, DataFrames, and MLlib.
- **`functions as F`** — the `pyspark.sql.functions` module contains all built-in column functions (`avg`, `count`, `round`, etc.). Aliasing it as `F` avoids name collisions with Python built-ins.
- **`StructType` / `StructField` / `*Type`** — used to define explicit DataFrame schemas, ensuring correct column types without relying on Spark's type inference.


In [7]:
# Install PySpark (only needed on Colab or environments without PySpark)
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # On Colab: install PySpark
    get_ipython().system('pip install pyspark --quiet')
    print("PySpark installed successfully.")
else:
    print("Not running on Colab — assuming PySpark is already installed.")

Not running on Colab — assuming PySpark is already installed.


<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 1: Regression — Apartment Price Prediction</h2>

We build a regression model to **predict apartment prices** based on:
- `City` — categorical feature (one of five Jordanian cities)
- `Bedrooms` — number of bedrooms
- `Bathrooms` — number of bathrooms
- `Square_Area` — size in square meters

We compare **Linear Regression** and **Decision Tree Regression**.

### 1.1 Generate the Apartment Dataset

We generate 120 synthetic apartment records using Python's `random` module, then load them into a Spark DataFrame.

- A **seed** (`random.seed(42)`) ensures the same data is produced every time the cell is run, making results reproducible.
- Each apartment is assigned a random city, number of bedrooms/bathrooms, and area. Price is derived from a realistic formula (base price per m² × area + room bonuses + Gaussian noise).
- `spark.createDataFrame(data, schema=...)` converts a plain Python list of tuples into a distributed Spark DataFrame. Providing an explicit **schema** (`StructType`) avoids Spark having to infer column types, which is both faster and safer.


In [8]:
import random

random.seed(42)

# City-specific base prices (JOD per m²) — same dataset as Notebook 3
city_base_price = {
    "Amman":  800,
    "Irbid":  500,
    "Zarqa":  400,
    "Aqaba":  600,
    "Madaba": 450,
}
cities = list(city_base_price.keys())

apartment_data = []
for _ in range(120):
    city       = random.choice(cities)
    bedrooms   = random.randint(1, 5)
    bathrooms  = random.randint(1, 3)
    area       = random.randint(45, 250)        # square meters
    noise      = random.gauss(0, 8000)          # price noise
    price      = (city_base_price[city] * area
                  + bedrooms * 3000
                  + bathrooms * 2000
                  + noise)
    price      = max(20000, round(price, -2))   # floor at 20 000 JOD
    apartment_data.append((city, bedrooms, bathrooms, area, float(price)))

# Define schema for the Spark DataFrame
apartments_schema = StructType([
    StructField("City",        StringType(),  True),
    StructField("Bedrooms",    IntegerType(), True),
    StructField("Bathrooms",   IntegerType(), True),
    StructField("Square_Area", IntegerType(), True),
    StructField("Price",       DoubleType(),  True),
])

apartments_df = spark.createDataFrame(apartment_data, schema=apartments_schema)

print(f"Dataset size: {apartments_df.count()} rows, {len(apartments_df.columns)} columns")
apartments_df.show(8, truncate=False)


NameError: name 'StructType' is not defined

### 1.2 Exploratory Data Analysis

Before building any model it is important to understand the data distribution. We do two things here:

1. **Summary statistics** — `describe()` returns count, mean, standard deviation, min, and max for each numeric column. This helps detect outliers and verify the data was generated correctly.
2. **Average price per city** — `groupBy("City").agg(avg("Price"))` computes per-city averages so we can verify that Amman (highest base price) is indeed the most expensive city on average, confirming our synthetic price formula is working as intended.


In [ ]:
# Summary statistics for numerical columns
print("=== Summary Statistics ===")
apartments_df.select("Bedrooms", "Bathrooms", "Square_Area", "Price").describe().show()

# Average price per city
print("=== Average Price by City ===")
apartments_df.groupBy("City") \
      .agg(F.round(F.avg("Price"), 0).alias("Avg_Price"),
           F.count("*").alias("Count")) \
      .orderBy(F.desc("Avg_Price")) \
      .show()


#### Pearson Correlation with Price

Pearson correlation (*r*) measures the **linear relationship** between two numeric columns. Values range from −1 (perfect negative linear relationship) to +1 (perfect positive linear relationship), with 0 indicating no linear relationship.

`apt_df.stat.corr("Square_Area", "Price")` calls Spark's built-in statistical correlation method — it is computed in a single distributed pass over the data. We expect `Square_Area` to have the highest correlation with `Price` because it directly multiplies the per-m² base price in our formula.


In [ ]:
# Pearson correlation of numeric features with Price
print("=== Correlation with Price ===")
for col in ["Bedrooms", "Bathrooms", "Square_Area"]:
    corr = apartments_df.stat.corr(col, "Price")
    print(f"  {col:15s}  r = {corr:.4f}")


### 1.3 Preprocessing — Encoding Categorical Features

MLlib algorithms require **all input to be numeric**. The `City` column contains strings, so we must convert it to numbers before feeding it to any model. We do this in two steps:

---

#### Step 1 — `StringIndexer`: String → Numeric Index

`StringIndexer` scans all distinct values in a column and assigns each a numeric index ordered by frequency (most frequent = 0).

```
"Amman"  →  0
"Aqaba"  →  1
"Irbid"  →  2
...
```

**Two-phase usage (fit → transform):**
- `.fit(dataframe)` — reads the data and **learns** the mapping (i.e. which city gets which index). Returns a fitted `StringIndexerModel`.
- `.transform(dataframe)` — **applies** the learned mapping to produce a new column `City_Index`.

This fit/transform pattern is used throughout MLlib so that the same mapping learned on training data can later be applied to unseen test data.

---

#### Step 2 — `OneHotEncoder`: Numeric Index → Binary Vector

Using `City_Index` directly would introduce a false ordering: the model could incorrectly infer that city 2 is "twice as much" as city 1. `OneHotEncoder` removes this by converting each index into a **sparse binary vector** with a 1 in the position that corresponds to that city and 0s everywhere else.

```
City_Index = 0  →  City_Vec = [1, 0, 0, 0]
City_Index = 1  →  City_Vec = [0, 1, 0, 0]
City_Index = 2  →  City_Vec = [0, 0, 1, 0]
...
```

*(Note: one category is dropped by default to avoid multicollinearity — this is the standard "dummy encoding" approach.)*

Again, `.fit()` learns the number of categories and `.transform()` applies the encoding.


In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# ========== ENCODING CATEGORICAL FEATURES: STRINGINDEXER + ONEHOTENCODER ==========
# MLlib algorithms require ALL inputs to be NUMERIC vectors. But we have 'City', which is a string.
# We must convert strings to numbers using a two-stage process: StringIndexer → OneHotEncoder
#
# STAGE 1: StringIndexer — Convert strings to numeric INDICES
# ──────────────────────────────────────────────────────────────
# StringIndexer looks at all distinct values and assigns each one a numeric index,
# ordered by frequency (most common = 0).

city_indexer      = StringIndexer(inputCol="City", outputCol="City_Index")
city_indexer_model = city_indexer.fit(apartments_df)   # Learn the City → Index mapping from data

# After learning, the mapping is:
#   "Amman"  → 0
#   "Irbid"  → 1
#   "Zarqa"  → 2
#   etc. (ordered by how often each city appears)

apartments_indexed = city_indexer_model.transform(apartments_df)  # Apply the learned mapping

# PROBLEM with using numeric indices directly:
# If we used City_Index directly in a model, the algorithm might think:
#   City_Index = 2 is "twice as much city" as City_Index = 1 ✗ WRONG!
# Numeric indices imply a FALSE ORDERING. We need to remove this.

# STAGE 2: OneHotEncoder — Convert numeric INDEX to SPARSE BINARY VECTOR
# ───────────────────────────────────────────────────────────────────────
# OneHotEncoder creates a "one-hot" (one position is 1, rest are 0) representation.
# This removes the false ordering because now each city is independent.

city_encoder      = OneHotEncoder(inputCol="City_Index", outputCol="City_Vec")
city_encoder_model = city_encoder.fit(apartments_indexed)   # Learn how many categories exist
apartments_encoded = city_encoder_model.transform(apartments_indexed)  # Apply encoding

# After OneHotEncoder, each city becomes a separate POSITION in the vector:
# City_Index = 0 (Amman)  → City_Vec = [1, 0, 0, 0] (one-hot)
# City_Index = 1 (Irbid)  → City_Vec = [0, 1, 0, 0]
# City_Index = 2 (Zarqa)  → City_Vec = [0, 0, 1, 0]
# City_Index = 3 (Aqaba)  → City_Vec = [0, 0, 0, 1]
#
# Now the model treats each city as INDEPENDENT, not as an ordered scale. ✓ CORRECT!

# Inspect the encoded columns
apartments_encoded.select("City", "City_Index", "City_Vec").show(6, truncate=False)


### 1.4 Feature Engineering — VectorAssembler

MLlib estimators expect a **single column** named `features` that contains all input values packed into one vector. `VectorAssembler` does exactly that: it takes a list of existing columns (numeric scalars or vectors) and merges them column-by-column into a single dense or sparse vector.

```
City_Vec   →  [1, 0, 0, 0]   (4 values from OHE)
Bedrooms   →  3
Bathrooms  →  1
Square_Area→  120
              ─────────────────────────────
features   →  [1.0, 0.0, 0.0, 0.0, 3.0, 1.0, 120.0]   (7-element vector)
```

`VectorAssembler` is a pure **Transformer** — it has no parameters to learn, so there is no `.fit()` step; calling `.transform()` is sufficient.

After assembly we keep only the `features` vector and the `Price` label (renamed to `label` as required by MLlib estimators).


In [ ]:
# Combine the OHE city vector + numeric features into one 'features' column
# VectorAssembler takes a list of input columns and merges them into a single dense/sparse
# vector column called 'features'. This single column is the format required by all MLlib estimators.
assembler = VectorAssembler(
    inputCols=["City_Vec", "Bedrooms", "Bathrooms", "Square_Area"],
    outputCol="features"
)

# ========== UNDERSTANDING VECTORASSEMBLER.TRANSFORM() ==========
# STEP-BY-STEP EXPLANATION:
# 1. We created the 'assembler' object above, which is a TRANSFORMER (a recipe for combining columns)
# 2. Now we APPLY ('transform') this recipe to the apartments_encoded DataFrame
# 3. The result is a NEW DataFrame with all original columns PLUS a new 'features' column
#    that contains a vector like: [1.0, 0.0, 0.0, 0.0, 3.0, 1.0, 120.0]
#    This vector packs: City_Vec(4 values) + Bedrooms + Bathrooms + Square_Area
# 4. Think of it like a RECIPE CARD:
#    - Assembler (created above) = the written recipe
#    - .transform() = actually making the dish using that recipe
#    - Result = the same ingredients, but now combined into one unified output
apartments_final = assembler.transform(apartments_encoded)

# Keep only features and label (Price)
apartments_ml = apartments_final.select("features", "Price").withColumnRenamed("Price", "label")

print("Feature vector example:")
apartments_ml.select("features", "label").show(5, truncate=False)


### 1.5 Train / Test Split

We split the prepared dataset into a **training set** (80 %) and a **test set** (20 %) using `randomSplit`.

- The model is trained exclusively on the training set — it never sees the test set during learning.
- The test set is held back to evaluate how well the model generalises to **new, unseen data**.
- Providing a fixed `seed` ensures the same split is produced each run, making comparisons between models fair and reproducible.


In [ ]:
# 80% training, 20% testing — seed for reproducibility
train_apartments, test_apartments = apartments_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training set : {train_apartments.count()} rows")
print(f"Test set     : {test_apartments.count()} rows")


### 1.6 Linear Regression

`LinearRegression` fits a straight-line (hyperplane) relationship between the feature vector and the continuous target (`Price`).

**Key hyperparameters used:**
| Parameter | Value | Meaning |
|---|---|---|
| `maxIter` | 100 | Maximum number of gradient-descent iterations |
| `regParam` | 0.01 | Regularisation strength — penalises large coefficients to reduce overfitting |
| `elasticNetParam` | 0 | Mix of L1/L2 penalty: 0 = pure **Ridge** (L2), 1 = pure **Lasso** (L1) |

**Evaluation metrics:**
- **RMSE** (Root Mean Squared Error) — average prediction error in the same units as the target (JOD). Lower is better.
- **R²** — fraction of variance in the target explained by the model. Ranges from 0 to 1; higher is better.


In [ ]:
# Instantiate and train Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,     # L2 regularisation strength
    elasticNetParam=0  # 0 = Ridge, 1 = Lasso
)

lr_model  = lr.fit(train_apartments)
lr_preds  = lr_model.transform(test_apartments)

# ========== UNDERSTANDING THE TWO-STEP EVALUATOR PATTERN ==========
# Many students ask: "Why do we create an evaluator, THEN call evaluate()?"
# It seems like extra work, but this design follows a key principle: SEPARATION OF CONCERNS
#
# STEP 1 — Create the Evaluator object (define the GRADING RUBRIC)
# ─────────────────────────────────────────────────────────────────────────
# RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
# This line says: "I want to calculate RMSE (Root Mean Squared Error) by comparing:
#   • the 'label' column (actual apartment prices from the test set)
#   • against the 'prediction' column (prices that the model predicted)
# Think of this as writing down the GRADING RUBRIC. It doesn't calculate anything yet.
# It just defines: WHAT metric to compute (RMSE), and WHERE to find the relevant columns (label vs prediction).

evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

# STEP 2 — Apply the evaluator to your predictions (APPLY THE GRADING RUBRIC)
# ─────────────────────────────────────────────────────────────────────────────
# evaluator_rmse.evaluate(lr_preds)
# This line gives the DataFrame lr_preds to the evaluator. The evaluator:
#   ✓ Looks at the 'label' column → actual apartment prices: [120000, 85000, 95000, ...]
#   ✓ Looks at the 'prediction' column → predicted prices: [118500, 87200, 93000, ...]
#   ✓ Computes RMSE using the formula: sqrt(mean((actual - predicted)²))
#   ✓ Returns a SINGLE NUMBER as the result (e.g., 5432.15 JOD)
# 
# WHY TWO STEPS?
# • It's cleaner to separate DEFINITION from EXECUTION
# • You can reuse the same evaluator for multiple models: evaluator.evaluate(model1), evaluate(model2), etc.
# • This design is used throughout Python data science (scikit-learn, MLlib, etc.)

lr_rmse = evaluator_rmse.evaluate(lr_preds)
lr_r2   = evaluator_r2.evaluate(lr_preds)

print(f"Linear Regression — RMSE : {lr_rmse:,.0f} JOD")
print(f"Linear Regression — R²   : {lr_r2:.4f}")

# Show sample predictions vs actuals
print("\nSample Predictions:")
lr_preds.select(
    F.round("label", 0).alias("Actual_Price"),
    F.round("prediction", 0).alias("Predicted_Price")
).show(8)


<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 2: Classification — Customer Churn Prediction</h2>

We predict whether a telecom customer will **churn** (cancel their subscription) using:
- `Age` — customer age
- `MonthlyCharges` — monthly bill in EUR
- `Tenure` — months as a customer
- `NumProducts` — number of subscribed products

Target: `Churned` (0 = stayed, 1 = churned)

### 2.1 Generate the Customer Churn Dataset

We generate 220 synthetic customer records. Each customer has four numeric attributes, and their `Churned` label (0 or 1) is generated probabilistically using a **logistic function**:

$$P(\text{churn}) = \frac{1}{1 + e^{-\text{logit}}}$$

where:

$$\text{logit} = -0.03 \times \text{tenure} + 0.02 \times \text{monthlyCharges} - 0.3 \times \text{numProducts} + 0.5$$

This formula encodes realistic business logic:
- Customers with **longer tenure** are less likely to churn (negative coefficient on `tenure`).
- Customers with **higher monthly charges** are more likely to churn (positive coefficient on `monthlyCharges`).
- Customers with **more products** are stickier and less likely to churn (negative coefficient on `numProducts`).

**Why not generate 0/1 directly?**
- The **logit** creates structured signal: different feature values produce different churn risk levels.
- The **random draw** adds realistic uncertainty: two similar customers can still have different outcomes.
- Together, this gives a **near-perfect but realistic** dataset: strongly learnable patterns with natural noise (closer to real business data than a fully deterministic rule).


In [ ]:
import random
import math

random.seed(7)

churn_data = []
for _ in range(220):
    age            = random.randint(18, 70)
    tenure         = random.randint(1, 72)          # months
    num_products   = random.randint(1, 5)
    monthly        = round(random.uniform(20.0, 150.0), 2)

    # ========== UNDERSTANDING PROBABILISTIC CHURN GENERATION ==========
    # We model churn using BUSINESS LOGIC encoded as a LOGISTIC REGRESSION FORMULA.
    # This formula encodes realistic customer behavior patterns:
    #   • Longer tenure → LESS likely to churn (coefficient -0.03 is negative)
    #   • Higher monthly charges → MORE likely to churn (coefficient +0.02 is positive)
    #   • More products → LESS likely to churn (coefficient -0.3, strongest effect)
    #   • Base tendency → +0.5 (slight bias toward churn before adjusting for features)
    #
    # The formula computes a "logit" (unbounded real number, can be positive or negative):
    logit  = (-0.03 * tenure
               + 0.02 * monthly
               - 0.3  * num_products
               + 0.5)
    # Examples:
    #   Customer A: tenure=60, monthly=50, num_products=4
    #     logit = (-0.03*60) + (0.02*50) - (0.3*4) + 0.5 = -1.8 + 1.0 - 1.2 + 0.5 = -1.5
    #
    #   Customer B: tenure=6, monthly=120, num_products=1
    #     logit = (-0.03*6) + (0.02*120) - (0.3*1) + 0.5 = -0.18 + 2.4 - 0.3 + 0.5 = 2.42
    #
    # Now we CONVERT the logit (a raw score) to a PROBABILITY (0 to 1) using the SIGMOID function:
    #   prob = 1 / (1 + e^(-logit))
    #
    # This sigmoid function has special properties:
    #   • If logit = 0,  prob = 0.5 (neutral, 50% chance)
    #   • If logit > 0,  prob > 0.5 (positive signal, higher churn risk)
    #   • If logit < 0,  prob < 0.5 (negative signal, lower churn risk)
    #
    # Examples from above:
    #   Customer A: logit = -1.5 → prob = 1/(1 + e^1.5) ≈ 0.18 (18% churn risk) ← LOW RISK
    #   Customer B: logit = 2.42 → prob = 1/(1 + e^-2.42) ≈ 0.92 (92% churn risk) ← HIGH RISK
    prob   = 1 / (1 + math.exp(-logit))
    #
    # WHY BOTH LOGIT AND RANDOM DRAW?
    # 1) logit/prob creates the SIGNAL (feature-driven risk pattern)
    # 2) random draw creates realistic NOISE (not everyone with same risk behaves identically)
    # This combination gives a near-perfect but realistic dataset.
    #
    # Finally, we CONVERT this probability to a BINARY OUTCOME (0 or 1):
    # We draw a random number between 0 and 1. If it's LESS THAN prob, the customer churns.
    # Otherwise, they stay.
    #
    # ANALOGY: Think of prob as a loaded coin:
    #   • prob = 0.7 means the coin has a 70% chance of landing HEADS (churn = 1)
    #   • prob = 0.3 means the coin has a 30% chance of landing HEADS (churn = 1)
    #   • We "flip the coin" by generating random.random() (0 to 1)
    #   • If the flip lands in [0, prob], we get churn = 1
    #   • If the flip lands in (prob, 1], we get churn = 0
    churned = 1 if random.random() < prob else 0

    churn_data.append((age, float(monthly), tenure, num_products, churned))

churn_schema = StructType([
    StructField("Age",            IntegerType(), True),
    StructField("MonthlyCharges", DoubleType(),  True),
    StructField("Tenure",         IntegerType(), True),
    StructField("NumProducts",    IntegerType(), True),
    StructField("Churned",        IntegerType(), True),
])

churn_df = spark.createDataFrame(churn_data, schema=churn_schema)

print(f"Dataset size: {churn_df.count()} rows")
churn_df.show(8)


### 2.2 Data Exploration and Class Distribution

Two key checks before modelling:

1. **Summary statistics** — `describe()` shows the range and spread of each feature. Verify that no feature has suspicious min/max values that would indicate data errors.
2. **Class distribution** — For classification problems, knowing whether classes are **balanced** (roughly equal counts of 0 and 1) or **imbalanced** matters. If one class dominates, accuracy alone is a misleading metric (e.g., always predicting "not churned" could achieve 80% accuracy while being useless). We compute the percentage of churned vs. retained customers to get a baseline.


In [ ]:
print("=== Summary Statistics ===")
churn_df.describe().show()

print("=== Class Distribution ===")
churn_df.groupBy("Churned") \
        .agg(F.count("*").alias("Count")) \
        .withColumn("Percentage",
                    F.round(F.col("Count") / churn_df.count() * 100, 1)) \
        .orderBy("Churned") \
        .show()

#### Pearson Correlation with Churn Label

We compute the Pearson correlation between each feature and the `Churned` binary label (0 or 1). While correlation measures only **linear** relationships, it still gives a useful preliminary signal:

- A **positive** correlation means the feature tends to increase together with churn probability.
- A **negative** correlation means higher values of the feature are associated with lower churn risk.

These values help confirm that the synthetic data reflects the business logic we encoded in the logistic formula used during generation.


In [ ]:
# Correlation of numeric features with churn label
print("=== Feature Correlations with Churn ===")
for col in ["Age", "MonthlyCharges", "Tenure", "NumProducts"]:
    corr = churn_df.stat.corr(col, "Churned")
    print(f"  {col:18s}  r = {corr:+.4f}")

### 2.3 Feature Engineering and Scaling

**`VectorAssembler`** merges the four numeric columns into a single `raw_features` vector (same concept as in Part 1).

**`StandardScaler`** then standardises each feature to **zero mean and unit variance**:

$$z = \frac{x - \mu}{\sigma}$$

This is important for **gradient-based algorithms** like Logistic Regression because:
- Features on very different scales (e.g., `Age` ∈ [18, 70] vs `MonthlyCharges` ∈ [20, 150]) cause the gradient to take much longer to converge — or to never converge.
- Scaling puts all features on equal footing so the regularisation penalty is applied fairly.

**Important fit/transform distinction:**
- `scaler.fit(churn_assembled)` computes the mean and standard deviation **from the training data only** and returns a `StandardScalerModel`.
- `scaler_model.transform(...)` applies those statistics to scale the data. When we later scale test data, we reuse the same `scaler_model` (trained statistics) rather than re-fitting — this prevents **data leakage** from the test set into the scaling parameters.


In [ ]:
from pyspark.ml.feature import StandardScaler

# Assemble all numeric features into a single vector
churn_assembler = VectorAssembler(
    inputCols=["Age", "MonthlyCharges", "Tenure", "NumProducts"],
    outputCol="raw_features"
)

# StandardScaler: zero mean, unit variance — important for Logistic Regression
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Apply transformations
churn_assembled = churn_assembler.transform(churn_df)

# ========== UNDERSTANDING FIT vs TRANSFORM: THE TWO-PHASE LEARNING PROCESS ==========
# StandardScaler (and most ML transformers) use a TWO-PHASE workflow: FIT then TRANSFORM
# This is because StandardScaler must LEARN statistics from the data before it can scale.
#
# PHASE 1: .fit() — LEARN statistics from the DATA
# ──────────────────────────────────────────────────────────────────────────────────
# scaler_model = scaler.fit(churn_assembled)
# This means:
#   ✓ Scan the entire churn_assembled DataFrame
#   ✓ Calculate the MEAN of each feature:
#       - Mean of Age: (18 + 25 + 70 + ...) / n = 44.2
#       - Mean of MonthlyCharges: (20 + 85 + 150 + ...) / n = 87.3
#       - Mean of Tenure: (1 + 6 + 72 + ...) / n = 36.8
#       - Mean of NumProducts: (1 + 2 + 5 + ...) / n = 2.7
#   ✓ Calculate the STANDARD DEVIATION (spread) of each feature:
#       - StdDev of Age: 16.5
#       - StdDev of MonthlyCharges: 32.1
#       - ... etc
#   ✓ STORE these learned statistics in a fitted object called scaler_model
#   ✓ Return scaler_model (a StandardScalerModel that "remembers" the mean and std)
#
# Why is this important? We learn from TRAINING data only, not test data!
# This prevents DATA LEAKAGE: test statistics should never influence preprocessing.
scaler_model    = scaler.fit(churn_assembled)

# PHASE 2: .transform() — APPLY the learned statistics to SCALE the data
# ──────────────────────────────────────────────────────────────────────
# churn_scaled = scaler_model.transform(churn_assembled)
# This means:
#   ✓ Take the scaler_model (which "remembers" the mean and std from Phase 1)
#   ✓ For EACH row in churn_assembled, apply the standardization formula:
#       z = (x - mean) / std
#   ✓ Examples:
#       - Age 45 with mean=44.2, std=16.5 → (45 - 44.2) / 16.5 ≈ 0.048 (near zero, close to mean)
#       - Age 25 with mean=44.2, std=16.5 → (25 - 44.2) / 16.5 ≈ -1.16 (below mean)
#       - Age 70 with mean=44.2, std=16.5 → (70 - 44.2) / 16.5 ≈ 1.57 (above mean)
#   ✓ Create a NEW DataFrame with a 'features' column containing scaled values
#
# RESULT: All features now have mean ≈ 0 and std ≈ 1 (standardized scale)
#
# ═══════════════════════════════════════════════════════════════════════════════════════════════
# WHY TWO SEPARATE STEPS?
# ─────────────────────────
# Training time:
#   • Fit on TRAINING data → learn mean/std from train
#   • Transform on TRAINING data → scale train using those statistics
#
# Test/Production time:
#   • REUSE the same scaler_model (fitted on training data)
#   • Transform on TEST data → scale test using TRAINING statistics, not test statistics
#   • This is CRUCIAL: if you re-fit on test data, test leaks into your preprocessing!
#
# Think of it like a language translator trained on English-Spanish text:
#   • .fit() = learn the vocabulary and grammar rules from training text
#   • .transform() = apply those rules to translate new text
#   • You DON'T re-learn the rules from the new text; you apply the learned rules
churn_scaled    = scaler_model.transform(churn_assembled)

# Rename label column to 'label' (required by MLlib estimators)
churn_ml = churn_scaled.select("features", F.col("Churned").cast(DoubleType()).alias("label"))

print("Prepared dataset sample:")
churn_ml.show(4, truncate=False)


### 2.3a Train / Test Split (Churn)

We split the scaled churn dataset into training (80 %) and test (20 %) sets using the same `randomSplit` approach as Part 1. The `seed=42` ensures a consistent split across runs so that model comparison results are reproducible.


In [ ]:
# Train / Test Split
train_churn, test_churn = churn_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Training set : {train_churn.count()} rows")
print(f"Test set     : {test_churn.count()} rows")

### 2.4 Logistic Regression

`LogisticRegression` models the **probability** that a customer churns using the logistic (sigmoid) function applied to a linear combination of features. It is a strong baseline for binary classification — fast, interpretable, and well-regularised.

**Key outputs produced by `.transform()`:**
| Column | Meaning |
|---|---|
| `rawPrediction` | The raw log-odds score (before sigmoid) for each class |
| `probability` | Softmax-transformed probabilities: `[P(0), P(1)]` |
| `prediction` | The class with the highest probability (0 or 1) |

**Confusion Matrix** — a 2×2 table comparing predicted vs actual classes:
- **True Positives (TP)**: predicted churn, actually churned ✓
- **True Negatives (TN)**: predicted stay, actually stayed ✓
- **False Positives (FP)**: predicted churn, actually stayed ✗
- **False Negatives (FN)**: predicted stay, actually churned ✗ (often the costliest mistake in churn prediction)


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Train Logistic Regression
log_reg = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

# Fit the Logistic Regression estimator on the training split
lr_cls_model = log_reg.fit(train_churn)
# Apply the fitted model to the test split to produce predictions
lr_cls_preds = lr_cls_model.transform(test_churn)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
# Compute classification accuracy on the test predictions
lr_accuracy = acc_evaluator.evaluate(lr_cls_preds)
print(f"Logistic Regression — Accuracy: {lr_accuracy:.4f}  ({lr_accuracy*100:.1f}%)")

# Building the confusion matrix requires THREE STEPS:

print("\nConfusion Matrix (Logistic Regression):")
print(f"{'':10s} {'Predicted 0':>14} {'Predicted 1':>14}")

# STEP 1: Group by both 'label' (actual) and 'prediction' (predicted), then count rows
# This creates a DataFrame like:
#   label | prediction | count
#   ──────┼────────────┼──────
#    0    │     0      │  42   (true negatives)
#    0    │     1      │   5   (false positives)
#    1    │     0      │   8   (false negatives)
#    1    │     1      │  19   (true positives)
cm = lr_cls_preds.groupBy("label", "prediction").count().collect()

# STEP 2: Convert the grouped results into a Python DICTIONARY for easy lookup
#
# Example: cm_dict[(0, 1)] = 5 means "5 customers with actual 0 got predicted 1"
#
# Dictionary comprehension syntax (advanced Python):
#   {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}
#   reads as: "for each row r in cm, create a key-value pair where key is the tuple
#   (label, prediction) and value is the count"
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}

# STEP 3: Print the confusion matrix as a formatted table
# Iterate through actual values (0 and 1), and for each one, show predictions (0 and 1)
for actual in [0, 1]:
    row = f"Actual {actual}   "
    for pred in [0, 1]:
        # .get((actual, pred), 0) means: "look up this key; if not found, return 0"
        # We use .get() because some combinations might have 0 count
        count = cm_dict.get((actual, pred), 0)
        row += f"{count:>14}"
    print(row)


<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 3: ML Pipelines</h2>

### What is a Pipeline and Why Use It?

A `Pipeline` encapsulates a sequence of data transformation and model training steps into a **single reusable object**.

**Key benefits:**
- **No data leakage**: preprocessing (e.g., scaling, encoding) is fitted *only* on training data, then applied to test data.
- **Reproducibility**: the exact same transformation sequence is always applied.
- **Deployment simplicity**: save one object; it contains both the transformations and the model.

### Pipeline Structure for This Example

```
VectorAssembler  →  StandardScaler  →  LogisticRegression
```

We build an end-to-end pipeline for churn classification: assemble features → scale them → train the classifier. All three steps are combined into one reproducible workflow.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Prepare raw churn data with a proper label column (no preprocessing yet)
churn_raw = churn_df.withColumn("label", F.col("Churned").cast(DoubleType()))

# Split the RAW data (the pipeline will handle all preprocessing internally)
train_raw, test_raw = churn_raw.randomSplit([0.8, 0.2], seed=42)

# --- Define Pipeline Stages ---

# Stage 1: Assemble numeric features into a raw vector
pipe_assembler = VectorAssembler(
    inputCols=["Age", "MonthlyCharges", "Tenure", "NumProducts"],
    outputCol="raw_features"
)

# Stage 2: Scale features (fitted on training data only — preventing data leakage)
pipe_scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Stage 3: Logistic Regression Classifier
pipe_lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

# Build the Pipeline object
pipeline = Pipeline(stages=[pipe_assembler, pipe_scaler, pipe_lr])

print("Pipeline stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Stage {i}: {type(stage).__name__}")


### Fit and Evaluate the Pipeline

Calling `pipeline.fit(train_raw)` triggers the **entire pipeline in sequence on the training data**:
1. `VectorAssembler.transform(train_raw)` → assembles the feature vector.
2. `StandardScaler.fit(assembled_train).transform(assembled_train)` → learns scaling parameters from training data only, then scales it.
3. `RandomForestClassifier.fit(scaled_train)` → trains the forest.

The result is a `PipelineModel` — a fitted version of the pipeline that remembers all learned parameters. When we call `pipeline_model.transform(test_raw)`, the same scaler statistics (not re-fitted) and the same forest are applied to the test data, preventing any data leakage.


In [ ]:
# Fit the pipeline on training data (all stages fit/transform in sequence)
pipeline_model = pipeline.fit(train_raw)

# Apply the fitted pipeline to test data
pipeline_preds = pipeline_model.transform(test_raw)

# Evaluate
pipe_accuracy = acc_evaluator.evaluate(pipeline_preds)
pipe_auc      = auc_evaluator.evaluate(pipeline_preds)

print(f"Pipeline Model — Accuracy : {pipe_accuracy:.4f}  ({pipe_accuracy*100:.1f}%)")
print(f"Pipeline Model — ROC-AUC  : {pipe_auc:.4f}")

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 5: Clustering — K-Means</h2>

**K-Means** is an unsupervised algorithm that partitions data into *k* clusters by minimizing the within-cluster sum of squared distances to each cluster centroid.

We generate a 2D dataset with **3 natural clusters** and use K-Means to recover them. We then apply the **Elbow Method** to choose the optimal *k*.

### 5.1 Generate 2D Clustered Dataset

We create a 2D synthetic dataset with **3 well-separated groups**. Using a low-dimensional (2D) dataset serves two purposes:
1. The true cluster structure is known, so we can verify that K-Means recovers it correctly.
2. Results are easy to reason about geometrically.

Each cluster centre is placed far from the others, and individual points are scattered around the centre using Gaussian noise (`random.gauss(0, 0.8)`). The `true_label` column stores the ground-truth cluster membership and will be used for visual verification — it is **not** passed to the K-Means algorithm (which is unsupervised).


In [ ]:
import random

random.seed(99)

# Three well-separated cluster centres
cluster_centres = [
    (2.0, 3.0),
    (8.0, 8.0),
    (5.0, 1.0),
]

cluster_data = []
for true_label, (cx, cy) in enumerate(cluster_centres):
    for _ in range(60):   # 60 points per cluster → 180 total
        x = cx + random.gauss(0, 0.8)
        y = cy + random.gauss(0, 0.8)
        cluster_data.append((float(x), float(y), true_label))

cluster_schema = StructType([
    StructField("x",          DoubleType(),  True),
    StructField("y",          DoubleType(),  True),
    StructField("true_label", IntegerType(), True),
])

cluster_df = spark.createDataFrame(cluster_data, schema=cluster_schema)

print(f"Cluster dataset: {cluster_df.count()} points across 3 clusters")
cluster_df.show(6)

### 5.2 K-Means with k=3

K-Means works iteratively:
1. **Initialise** — place *k* centroids (randomly or using k-means++ seeding).
2. **Assign** — assign each point to its nearest centroid (Euclidean distance).
3. **Update** — move each centroid to the mean of all points assigned to it.
4. Repeat steps 2–3 until centroids stop moving (convergence) or `maxIter` is reached.

**`VectorAssembler`** first packs the `x` and `y` columns into a single `features` vector, which is what `KMeans` requires.

**Silhouette Score** measures how well-separated the clusters are:
- Score near **+1** → points are tightly packed within their cluster and far from others.
- Score near **0** → clusters overlap significantly.
- Score near **-1** → many points are closer to a different cluster than their own.


In [ ]:
# --- Train and Evaluate a Simple Logistic Regression Model ---
# Import the needed tools for logistic regression and evaluation
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Set up the logistic regression model (like picking a recipe)
log_reg = LogisticRegression(
    featuresCol="features",  # Where our input numbers are
    labelCol="label",      # Where our answers (0/1) are
    maxIter=100,            # Try up to 100 times to find the best fit
    regParam=0.01           # Small penalty to avoid overfitting
 )

# Train the model using the training data (let the computer learn)
lr_cls_model = log_reg.fit(train_churn)

# Use the trained model to make predictions on the test data
lr_cls_preds = lr_cls_model.transform(test_churn)

# --- Check How Well the Model Did (Accuracy) ---
# Set up the accuracy checker
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
 )
# Calculate accuracy (how many predictions were correct)
lr_accuracy = acc_evaluator.evaluate(lr_cls_preds)
print(f"Logistic Regression — Accuracy: {lr_accuracy:.4f}  ({lr_accuracy*100:.1f}%)")

# --- Build and Show the Confusion Matrix (Who was right/wrong) ---
# The confusion matrix shows how many times the model got each type of answer:
# - True Negative (TN): Predicted 0, Actually 0
# - False Positive (FP): Predicted 1, Actually 0
# - False Negative (FN): Predicted 0, Actually 1
# - True Positive (TP): Predicted 1, Actually 1

print("\nConfusion Matrix (Logistic Regression):")
print(f"{'':10s} {'Predicted 0':>14} {'Predicted 1':>14}")

# Count how many times each (actual, predicted) pair happened
cm = lr_cls_preds.groupBy("label", "prediction").count().collect()

# Put the counts in a dictionary for easy lookup
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}

# Print the confusion matrix as a table
for actual in [0, 1]:
    row = f"Actual {actual}   "
    for pred in [0, 1]:
        count = cm_dict.get((actual, pred), 0)  # 0 if not found
        row += f"{count:>14}"
    print(row)

### 5.3 Elbow Method — Choosing the Optimal k

Choosing the right number of clusters *k* is one of the central challenges in unsupervised learning. The **Elbow Method** helps by running K-Means for a range of *k* values and recording the **Within-Cluster Sum of Squared Errors (WCSSE)** — the total squared distance between each point and its assigned centroid.

- Adding more clusters always reduces WCSSE (in the extreme, k = n gives WCSSE = 0).
- The goal is to find the **"elbow"** — the *k* where WCSSE drops steeply before but only marginally after. Beyond the elbow, additional clusters are splitting already-tight groups and provide little real benefit.

`model.summary.trainingCost` returns the WCSSE directly from the fitted `KMeansModel`.

We pair the WCSSE values with the **Silhouette Score** for a more reliable choice: the optimal *k* should have both a low WCSSE (tight clusters) and a high Silhouette Score (well-separated clusters).


In [ ]:
# ========== THE ELBOW METHOD: CHOOSING THE OPTIMAL k ==========
#
# PROBLEM: How do we choose k (the number of clusters)?
# • Too few clusters → important groups get merged, lose information
# • Too many clusters → break apart natural groups, waste resources
#
# SOLUTION: The ELBOW METHOD
# Run K-Means for many different k values and watch how WCSSE (Within-Cluster Sum of Squared Errors) changes.
#
# WCSSE = sum of squared distances from each point to its assigned cluster centre
#   • Lower WCSSE = tighter, more compact clusters = better fit
#   • Adding more clusters ALWAYS reduces WCSSE (in extreme, k=n gives WCSSE=0)
#
# Plot WCSSE vs k: usually we see an ELBOW pattern
#   • k=2 to k=3: WCSSE drops STEEPLY (big improvement)
#   • k=3 to k=4: WCSSE drops LESS (smaller improvement)
#   • k=4 to k=5: WCSSE drops MARGINALLY (almost no improvement)
#   • The ELBOW (where the curve bends) indicates the sweet spot
#
# At the elbow, adding more clusters doesn't help much, so we stop there.

# Evaluate K-Means for k = 2 to 7 and record WCSSE and Silhouette score
elbow_results = []

for k in range(2, 8):
    # Train K-Means with this k value
    km_k      = KMeans(featuresCol="features", predictionCol="cluster",
                       k=k, seed=42, maxIter=20)
    model_k   = km_k.fit(cluster_features)
    preds_k   = model_k.transform(cluster_features)
    
    # WCSSE: sum of squared distances (lower = better fit)
    # model_k.summary.trainingCost gives the WCSSE directly
    wcsse     = model_k.summary.trainingCost
    
    # Silhouette: how well-separated are the clusters? (-1 to +1, higher = better)
    sil_k     = sil_evaluator.evaluate(preds_k)
    
    elbow_results.append((k, round(wcsse, 2), round(sil_k, 4)))
    print(f"  k={k}  WCSSE={wcsse:>10,.2f}  Silhouette={sil_k:.4f}")

print("\nDone.")


#### ASCII Bar Chart — Elbow Visualisation

Since we are working inside a notebook without a plotting library, we render the Elbow Method results as a simple **text-based bar chart**. Each bar's length is proportional to the WCSSE value relative to the maximum WCSSE (k=2). The elbow at k=3 is annotated with an arrow marker. This makes the result immediately readable without requiring Matplotlib or any external library.


In [ ]:
# ASCII bar chart for visual inspection of the Elbow Method
print("\nElbow Method — WCSSE by k")
print("=" * 55)
max_wcsse = max(r[1] for r in elbow_results)
for k, wcsse, sil in elbow_results:
    bar_len = int((wcsse / max_wcsse) * 35)
    bar     = "█" * bar_len
    marker  = " ← elbow" if k == 3 else ""
    print(f"  k={k}  {bar:<36} {wcsse:>10,.0f}{marker}")
print("=" * 55)

### Elbow Method Results Summary

| k | WCSSE (lower = better fit) | Silhouette Score (higher = better separation) |
|---|---|---|
| 2 | High | Low–Medium |
| 3 | **Elbow point — large drop from k=2** | **Highest** |
| 4 | Moderate decrease | Decreasing |
| 5 | Small decrease | Further decrease |
| 6 | Marginal decrease | Low |
| 7 | Marginal decrease | Low |

The elbow at **k=3** corresponds to our 3 natural clusters, confirming the method works correctly on this dataset.

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Summary — Algorithms Covered</h2>

| Algorithm | Type | MLlib Class | Key Hyperparameters | When to Use |
|---|---|---|---|---|
| **Linear Regression** | Regression | `LinearRegression` | `maxIter`, `regParam`, `elasticNetParam` | Continuous target, features have roughly linear relationship with target |
| **Logistic Regression** | Classification | `LogisticRegression` | `maxIter`, `regParam` | Binary or multiclass classification; fast, interpretable, good baseline |
| **K-Means** | Clustering | `KMeans` | `k`, `maxIter`, `tol` | Discover natural groupings in unlabelled data; works best with spherical clusters |
| **Pipeline** | Workflow | `Pipeline` | — | Combine preprocessing + model into a single reusable, leak-free object |

### Key Takeaways

- Always use **Pipelines** in production — they prevent data leakage and simplify deployment.
- **StandardScaler** is essential for gradient-based algorithms (Logistic Regression, K-Means).
- For classification, use **ROC-AUC** alongside accuracy — accuracy can be misleading on imbalanced datasets.
- The **Elbow Method** and **Silhouette Score** together guide the choice of *k* in K-Means.
- **MLlib API is similar to scikit-learn**, making it easy to transition from single-machine to distributed ML.


### Stopping the SparkSession

When you are finished with all computations, call `spark.stop()` to **release all resources** held by the Spark driver and executors (memory, threads, and any temporary files). This is especially important when running locally to avoid memory leaks between notebook sessions.

After `spark.stop()`, any further Spark operations will fail unless a new `SparkSession` is created.


In [ ]:
# Cleanly stop the SparkSession when done
spark.stop()
print("SparkSession stopped.")